# Week 10 extension — NLL ablations across all experiments (notebook 10e)

This is the evaluation half of the Week 10 extension. It scores every
checkpoint produced by 10d (`ckpt_E*.ckpt` in this directory) against
the same **hard-gated NLL** metric used in 10c, and adds a critical new
diagnostic: an **oracle MLP** that maps each experiment's cond vector
directly to per-bin Gaussian residual parameters. The oracle's NLL is
an *upper bound* on what any model can extract from a given cond set:

- Diffusion ≪ oracle  →  architecture is the bottleneck (try FiLM, CFG, Fourier).
- Oracle ≈ classical  →  the cond set itself doesn't help; try a different
  cond group or stop running that variant.

This is what makes the ablation scientifically honest: without an
oracle, a flat NLL across experiments could mean *either* "more cond
doesn't help" *or* "the architecture can't extract the new cond's
information" — two completely different fixes.

**The test split is reserved for the PI.** Every cell in this notebook
filters to `split in {"train", "val"}`.


In [ ]:
# Standard setup.
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import norm as sp_norm
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat


In [ ]:
# ── locate artifacts (same discovery pattern as 10c) ──────────────────────
def _find(filename, search_dirs):
    for d in search_dirs:
        p = os.path.join(d, filename)
        if os.path.isfile(p):
            return p
    return None

_cwd = os.getcwd()
_search_dirs = []
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    for _sub in [("weeks", "week_10"), ("weeks", "week_09"), ("weeks", "week_08")]:
        _candidate = os.path.join(_base, *_sub)
        if os.path.isdir(_candidate) and _candidate not in _search_dirs:
            _search_dirs.append(_candidate)
    if (any(w in _base for w in ("week_08", "week_09", "week_10"))
            and os.path.isdir(_base) and _base not in _search_dirs):
        _search_dirs.append(_base)

_unconditioned_py  = _find("unconditioned_infrastructure.py", _search_dirs)
_conditioned_py    = _find("conditioned_infrastructure.py",   _search_dirs)
_parquet_v2_path   = _find("diffusion_windows_v2.parquet",    _search_dirs)
_classical_py      = _find("butterflAI_model.py",             _search_dirs)
_classical_weights = _find("official_model.npz",              _search_dirs)
_ckpt_cond_w10     = _find("ckpt_conditional.ckpt",           _search_dirs)
_raw_csv_path      = None
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(0, 5)]:
    _c = os.path.join(_base, "data", "composite_sunspot_groups_peak_area.csv")
    if os.path.isfile(_c):
        _raw_csv_path = _c; break

_missing = [n for n, p in [
    ("unconditioned_infrastructure.py", _unconditioned_py),
    ("conditioned_infrastructure.py",   _conditioned_py),
    ("diffusion_windows_v2.parquet",    _parquet_v2_path),
    ("butterflAI_model.py",             _classical_py),
    ("official_model.npz",              _classical_weights),
    ("data/composite_sunspot_groups_peak_area.csv", _raw_csv_path),
] if p is None]
if _missing:
    raise FileNotFoundError(
        f"Cannot locate {_missing}. Did you run 10d Part A to build the v2 parquet?"
    )

_repo_root = os.path.abspath(os.path.join(os.path.dirname(_conditioned_py), "..", ".."))
for _p in [_repo_root,
           os.path.dirname(_unconditioned_py),
           os.path.dirname(_conditioned_py),
           os.path.dirname(_classical_py)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

if "conditioned_infrastructure" in sys.modules:
    _existing = sys.modules["conditioned_infrastructure"]
    if getattr(_existing, "__file__", None) != _conditioned_py:
        del sys.modules["conditioned_infrastructure"]

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    ExtendedConditionalDiffusionLightning,
    sample_conditional_extended,
    build_model,
)
from butterflAI_model import ButterflAIModel

classical  = ButterflAIModel(_classical_weights)
windows_v2 = pd.read_parquet(_parquet_v2_path)
# Test split is reserved for the PI.
windows_v2 = windows_v2.loc[windows_v2["split"].isin(["train", "val"])].reset_index(drop=True)

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_WEEK10_DIR = os.path.dirname(_conditioned_py)
print(f"v2 parquet (train+val): {len(windows_v2)} rows")
print(f"checkpoints dir       : {_WEEK10_DIR}")
print(f"device                : {device}")
print(classical)


---
## The experiment menu (must match 10d)

The dict below mirrors the one in 10d. The eval loop looks up each
discovered `ckpt_E*.ckpt` by name in this dict to know which cond
groups to assemble, which arch to instantiate, and (for E6) whether to
sweep the guidance weight.

If you redefined any experiment in 10d, mirror the change here.


In [ ]:
# Same experiment menu as 10d. Keep these in sync.
_BASE_TEMPLATE = {
    "arch":          "concat",
    "consumed_keys": ["cond_base"],
    "groups":        ["base"],
    "hidden_dim":    128,
    "n_layers":      3,
    "fourier":       False,
    "cond_dropout_p": 0.0,
    "max_epochs":    20000,
    "lr":            1e-3,
    "batch_size":    64,
}
def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    "E0": _spec(),
    "E1": _spec(consumed_keys=["cond_base", "cond_cyclehemi"], groups=["base", "cyclehemi"]),
    "E2": _spec(consumed_keys=["cond_base", "cond_opp"],       groups=["base", "opp"]),
    "E3": _spec(consumed_keys=["cond_base", "cond_traj"],      groups=["base", "traj"]),
    "E4": _spec(arch="film"),
    "E5": _spec(arch="film", consumed_keys=["cond_base", "cond_opp"], groups=["base", "opp"]),
    "E6": _spec(arch="film", consumed_keys=["cond_base", "cond_opp"], groups=["base", "opp"],
                cond_dropout_p=0.1),
    "E7": _spec(arch="film", consumed_keys=["cond_base", "cond_opp"], groups=["base", "opp"],
                fourier=True),
}

# Which guidance values to sweep when scoring a CFG-trained checkpoint.
CFG_GUIDANCE_W = [1.0, 1.5, 2.0, 3.0]

# Discover trained checkpoints.
_ckpt_paths = sorted(glob.glob(os.path.join(_WEEK10_DIR, "ckpt_E*.ckpt")))
_discovered = [os.path.splitext(os.path.basename(p))[0].replace("ckpt_", "")
               for p in _ckpt_paths]
print(f"discovered checkpoints: {_discovered}")
for name in _discovered:
    assert name in EXPERIMENTS, f"ckpt_{name}.ckpt has no entry in EXPERIMENTS"


---
## Task 67 — Build per-window evaluation blocks

Re-window the raw CSV the same way 10c does (per-window, 6-monthly,
≥ 20 obs per window) and tag each window with its v2 parquet row's
*entire* cond superset — every group, normalized later per-experiment
using the corresponding checkpoint's `cond_<g>_means` / `cond_<g>_stds`
buffers. We work with **per-window blocks only** in 10e — that is the
granularity at which the diffusion model is native, and the granularity
where any improvement over Week 10 will be most visible.


In [ ]:
# Task 67 — assemble per-window blocks tagged with their v2 cond superset.

raw_df = pd.read_csv(_raw_csv_path)
raw_df["date"]       = pd.to_datetime(raw_df[["year", "month", "day"]])
raw_df["abs_lat"]    = raw_df["latitude"].abs()
raw_df["hemisphere"] = np.where(raw_df["latitude"] >= 0, "north", "south")
raw_df = raw_df.dropna(subset=["CYCLE"]).copy()
raw_df["CYCLE"]      = raw_df["CYCLE"].astype(int)
raw_df["year"]       = raw_df["date"].dt.year

# Group cols in the v2 parquet — must mirror ExtendedConditionalResidualDataset.GROUP_COLS.
GROUP_COLS = {
    "base":      ["area_smoothed", "mu_universal", "model_sigma", "amplitude"],
    "cyclehemi": ["cycle_norm", "hemi_id"],
    "opp":       ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"],
    "traj":      sorted([c for c in windows_v2.columns if c.startswith("area_lag")],
                        key=lambda c: int(c.replace("area_lag", ""))),
}
print("group cols resolved:", {g: c for g, c in GROUP_COLS.items()})

hc_split   = (windows_v2.groupby(["cycle", "hemisphere"])["split"]
                         .agg(lambda s: s.iloc[0]).to_dict())
amp_lookup = (windows_v2.groupby(["cycle", "hemisphere"])["amplitude"]
                         .agg("first").to_dict())
_t0_by_hc  = {(int(c), str(h)): float(classical.lookup_t0(int(c), h))
              for (c, h) in classical.known_hemicycles()}

# Pre-compute each parquet row's year_center (for closest-center lookup).
_wdf = windows_v2.copy()
_wdf = _wdf[_wdf.apply(lambda r: (int(r["cycle"]), str(r["hemisphere"])) in _t0_by_hc, axis=1)]
_wdf["year_center"] = _wdf.apply(
    lambda r: float(r["tau_center"]) + _t0_by_hc[(int(r["cycle"]), str(r["hemisphere"]))],
    axis=1,
)

# Per-window cond lookup: dict keyed by (cycle, hemi, year_center) → dict
# of group → raw (unnormalized) vector. We normalize later using each
# checkpoint's per-group buffers.
cond_lookup_window = {}
for _, r in _wdf.iterrows():
    key = (int(r["cycle"]), str(r["hemisphere"]), float(r["year_center"]))
    cond_lookup_window[key] = {g: np.array([float(r[c]) for c in GROUP_COLS[g]],
                                            dtype=np.float32)
                                for g in GROUP_COLS}


def _lookup_cond(cyc, hemi, c_dec, tol=1e-2):
    best, best_d = None, tol
    for (c2, h2, yc), v in cond_lookup_window.items():
        if c2 != cyc or h2 != hemi: continue
        d = abs(yc - c_dec)
        if d <= best_d: best, best_d = v, d
    return best


def build_per_window_hc(cyc, hemi, df_hc):
    if len(df_hc) == 0: return []
    y0, y1 = df_hc["date"].min().year, df_hc["date"].max().year + 1
    bounds = sorted({pd.Timestamp(y, m, 1) for y in range(y0, y1 + 1) for m in (1, 7)})
    out = []
    for ws, we in zip(bounds[:-1], bounds[1:]):
        mask = (df_hc["date"] >= ws) & (df_hc["date"] < we)
        dfw  = df_hc.loc[mask]
        if len(dfw) < 20: continue
        c_dec = ws.year + ws.dayofyear / 365.25 + (we - ws).days / (2 * 365.25)
        groups = _lookup_cond(cyc, hemi, c_dec)
        if groups is None: continue
        out.append({"center_decimal": float(c_dec),
                    "tau":  float(classical.to_tau(cyc, hemi, c_dec)),
                    "lats": dfw["abs_lat"].to_numpy(np.float32),
                    "groups_raw": groups})
    return out


hemicycles = []
for (cyc, hemi), split in hc_split.items():
    if split not in ("train", "val"): continue
    try:    t0 = float(classical.lookup_t0(cyc, hemi))
    except KeyError: continue
    A    = float(amp_lookup[(cyc, hemi)])
    dfh  = raw_df[(raw_df["CYCLE"] == int(cyc)) & (raw_df["hemisphere"] == hemi)]
    blks = build_per_window_hc(cyc, hemi, dfh)
    if not blks: continue
    hemicycles.append({"cycle": int(cyc), "hemisphere": hemi, "amplitude": A,
                       "t0": t0, "split": split, "blocks": blks})

parq_n  = ((windows_v2["split"] == "train") | (windows_v2["split"] == "val")).sum()
built_n = sum(len(hc["blocks"]) for hc in hemicycles)
print(f"per-window blocks built: {built_n} (parquet train+val: {parq_n})")
print(f"hemicycles included    : {len(hemicycles)}")


---
## Task 67 (cont) — NLL primitives, same as 10c

These are byte-identical to the primitives in 10c. Re-stated here so
10e can be run standalone (without executing 10c first).


In [ ]:
# Task 67 — hard NLL primitives (copied from 10c so this notebook stands alone).

def hard_nll_classical(model, hcs):
    total, included, candidate = 0.0, 0, 0
    for hc in hcs:
        A    = hc["amplitude"]; mu0A = float(model.mu_0(A))
        for blk in hc["blocks"]:
            candidate += 1
            mu = float(model.mu(blk["tau"]))
            if mu > mu0A: continue
            sigma = float(model.sigma(mu, A))
            if sigma <= 0: continue
            ll = sp_norm.logpdf(blk["lats"], loc=mu, scale=sigma).mean()
            if not np.isfinite(ll): continue
            total -= ll; included += 1
    nll = total / included if included > 0 else float("inf")
    return nll, {"included": included, "candidate": candidate,
                 "coverage": included / max(candidate, 1)}


def hard_nll_combined(model, hcs, residuals_by_block, eps=1e-6):
    total, included, candidate = 0.0, 0, 0
    n_floored, n_lats = 0, 0
    for hc in hcs:
        A    = hc["amplitude"]; mu0A = float(model.mu_0(A))
        for blk in hc["blocks"]:
            candidate += 1
            mu = float(model.mu(blk["tau"]))
            if mu > mu0A: continue
            sigma = float(model.sigma(mu, A))
            if sigma <= 0: continue
            key = (hc["cycle"], hc["hemisphere"], blk["center_decimal"])
            if key not in residuals_by_block: continue
            residual = residuals_by_block[key]
            lats   = blk["lats"]
            p_cl   = sp_norm.pdf(lats, loc=mu, scale=sigma)
            bin_ix = np.clip(np.floor(lats / BIN_WIDTH).astype(int), 0, 14)
            p_raw  = p_cl + residual[bin_ix]
            p_comb = np.maximum(eps, p_raw)
            n_floored += int((p_raw < eps).sum()); n_lats += len(lats)
            ll = np.log(p_comb).mean()
            if not np.isfinite(ll): continue
            total -= ll; included += 1
    nll = total / included if included > 0 else float("inf")
    return nll, {"included": included, "candidate": candidate,
                 "coverage": included / max(candidate, 1),
                 "floor_fraction": n_floored / max(n_lats, 1)}


# Classical baseline on the val split — the number to beat.
val_hcs = [hc for hc in hemicycles if hc["split"] == "val"]
nll_cl_val, det_cl_val = hard_nll_classical(classical, val_hcs)
print(f"classical hard NLL (val): {nll_cl_val:.4f}  (coverage {det_cl_val['coverage']:.3f})")


---
## Task 67 (cont) — Diagnostic oracle

For each experiment's cond set, fit a tiny MLP that maps the cond
vector directly to a 15-D Gaussian over the residual bins
(`mean`, `log_std`). The oracle's NLL is computed by sampling K
residuals from the per-block Gaussian and feeding them through
`hard_nll_combined` — exactly the same harness used to score the
diffusion.

What the oracle answers:

- **Diffusion ≪ oracle**  →  the diffusion isn't extracting the
  information that's already in the cond. Try a stronger architecture
  (FiLM, Fourier features, larger MLP).
- **Oracle ≈ classical**  →  the cond set itself doesn't carry enough
  information about the residual structure. Try a different cond
  group or stop adding to this one.

You implement this. The harness below provides the data loaders and
the eval glue; the science (the tiny MLP, the Gaussian NLL training
loop, sampling from the predicted Gaussian) is yours.


In [ ]:
# Task 67 — the oracle MLP. Students fill in the body.

class OracleMLP(nn.Module):
    """Map a cond vector to per-bin Gaussian residual params (mean, log_std).

    TODO (students): implement a small MLP (2-3 hidden layers, SiLU).
    Output is 2 * 15 = 30 numbers per row: 15 means + 15 log-stds.
    """
    def __init__(self, cond_dim, hidden_dim=64):
        super().__init__()
        # TODO: build the layers.
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, 2 * 15),
        )
    def forward(self, cond):
        out = self.net(cond)                              # (B, 30)
        mean, log_std = out[:, :15], out[:, 15:]
        return mean, log_std


def gaussian_nll(r, mean, log_std):
    """Per-row, per-bin Gaussian NLL of the *standardized* residual ``r``
    under the predicted (mean, log_std). Returns scalar."""
    # TODO (students): replace this with the closed-form expression.
    var = (2 * log_std).exp()
    return (0.5 * ((r - mean) ** 2) / var + log_std + 0.5 * np.log(2 * np.pi)).mean()


def fit_oracle(cond_train, r_train, cond_val, r_val,
               max_epochs=500, lr=1e-2, hidden_dim=64, seed=0):
    """Fit OracleMLP on (cond_train, r_train); return the trained module
    and the val NLL it achieves. r_* are *standardized* residuals."""
    torch.manual_seed(seed)
    cond_dim = cond_train.shape[1]
    oracle   = OracleMLP(cond_dim=cond_dim, hidden_dim=hidden_dim).to(device)
    opt      = torch.optim.Adam(oracle.parameters(), lr=lr, weight_decay=1e-4)
    best_val = float("inf"); best_state = None
    for ep in range(max_epochs):
        oracle.train()
        mean, log_std = oracle(cond_train)
        loss = gaussian_nll(r_train, mean, log_std)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            oracle.eval()
            mv, lv = oracle(cond_val)
            vloss  = gaussian_nll(r_val, mv, lv).item()
            if vloss < best_val:
                best_val = vloss
                best_state = {k: v.detach().clone() for k, v in oracle.state_dict().items()}
    if best_state is not None:
        oracle.load_state_dict(best_state)
    return oracle, best_val


---
## Task 68 — Score every checkpoint

For each discovered `ckpt_E*.ckpt`:

1. Build per-experiment cond tensors for every val block by
   concatenating the right groups in `consumed_keys` order, normalized
   with the **checkpoint's own** per-group buffers (so val data uses
   train-set normalization recovered from the saved model).
2. Run K = 100 conditional samples per block using
   `sample_conditional_extended`. For E6 (CFG), repeat the sampling at
   every guidance weight in `CFG_GUIDANCE_W` and keep them as separate
   rows.
3. Fit the oracle MLP on the same (cond, standardized residual) data
   and record its val NLL as the upper bound for this cond set.
4. Plug each of the K samples into `hard_nll_combined`; report mean
   and σ over K.


In [ ]:
# Task 68 — score loop.

K = 100


def _load_checkpoint(name, cfg):
    """Rebuild the dataset (to discover total cond dim and group_stats)
    and load the matching checkpoint into an ExtendedConditionalDiffusionLightning."""
    train_ds = ExtendedConditionalResidualDataset(windows_v2, "train", groups=cfg["groups"])
    val_ds   = ExtendedConditionalResidualDataset(windows_v2, "val",   groups=cfg["groups"],
                                                  group_stats=train_ds.group_stats)
    total_dim = sum(train_ds.group_dims[k.replace("cond_", "")] for k in cfg["consumed_keys"])

    model = build_model(cfg, total_cond_dim=total_dim)
    lit   = ExtendedConditionalDiffusionLightning.load_from_checkpoint(
        os.path.join(_WEEK10_DIR, f"ckpt_{name}.ckpt"),
        model=model, alpha=alpha_np, sigma=sigma_np,
        group_stats={g: train_ds.group_stats[g] for g in cfg["groups"]
                     if f"cond_{g}" in cfg["consumed_keys"]},
        total_cond_dim=total_dim, consumed_keys=cfg["consumed_keys"],
        map_location=device,
    ).to(device).eval()
    return lit, train_ds, val_ds, total_dim


def _block_cond_concat(hcs, lit, cfg, train_ds):
    """For each block in ``hcs``, build the concatenated, normalized cond
    vector in ``cfg['consumed_keys']`` order. Returns (keys, cond_tensor)."""
    keys, rows = [], []
    for hc in hcs:
        for blk in hc["blocks"]:
            keys.append((hc["cycle"], hc["hemisphere"], blk["center_decimal"]))
            parts = []
            for k in cfg["consumed_keys"]:
                g     = k.replace("cond_", "")
                raw   = torch.tensor(blk["groups_raw"][g], dtype=torch.float32)
                means = getattr(lit, f"cond_{g}_means").cpu()
                stds  = getattr(lit, f"cond_{g}_stds").cpu()
                parts.append((raw - means) / stds)
            rows.append(torch.cat(parts, dim=-1))
    return keys, torch.stack(rows, dim=0)


def _k_run_combined(hcs, keys, sample_NK15):
    nlls, floors = [], []
    for k in range(sample_NK15.shape[1]):
        rbb = {key: sample_NK15[i, k] for i, key in enumerate(keys)}
        nll, det = hard_nll_combined(classical, hcs, rbb)
        nlls.append(nll); floors.append(det["floor_fraction"])
    return np.asarray(nlls), np.asarray(floors)


def _oracle_for(name, cfg, val_hcs, lit, train_ds, val_ds):
    """Train the oracle on TRAIN cond/residuals, eval per-K on val blocks."""
    # Build (cond, standardized residual) tensors from the dataset.
    def _tensors(ds, hcs_split):
        # Build cond from the val blocks (so we pair with raw lats later)
        keys, cond = _block_cond_concat(hcs_split, lit, cfg, train_ds)
        # r_clean per block: use the dataset row whose tau_center matches
        # the block center. The dataset rows are aligned with windows_v2.
        r_clean = []
        for key in keys:
            cyc, hemi, cd = key
            row = windows_v2.loc[
                (windows_v2["cycle"] == cyc) &
                (windows_v2["hemisphere"] == hemi)
            ]
            # closest in year_center.
            t0 = _t0_by_hc[(cyc, hemi)]
            ycs = row["tau_center"].to_numpy() + t0
            ix  = int(np.argmin(np.abs(ycs - cd)))
            emp = row.iloc[ix][[f"hist_emp_{j:02d}" for j in range(15)]].to_numpy(np.float32)
            par = row.iloc[ix][[f"hist_par_{j:02d}" for j in range(15)]].to_numpy(np.float32)
            r_clean.append(torch.from_numpy(emp - par))
        if not r_clean:
            return cond, torch.empty(0, 15)
        r_raw = torch.stack(r_clean, dim=0)
        # Standardize using TRAIN bin stats.
        r_std = (r_raw - train_ds.bin_means) / train_ds.bin_stds
        return cond, r_std

    train_hcs = [hc for hc in hemicycles if hc["split"] == "train"]
    c_tr, r_tr = _tensors(train_ds, train_hcs)
    c_va, r_va = _tensors(val_ds,   val_hcs)
    c_tr, r_tr, c_va, r_va = (x.to(device) for x in (c_tr, r_tr, c_va, r_va))

    oracle, val_gauss_nll = fit_oracle(c_tr, r_tr, c_va, r_va,
                                       max_epochs=600, lr=1e-2, hidden_dim=64)

    # Convert oracle's per-block Gaussian into K samples in *physical*
    # residual units, push through hard_nll_combined.
    with torch.no_grad():
        mean, log_std = oracle(c_va)
        std = log_std.exp()
        eps = torch.randn(K, *mean.shape, device=device)
        samples_std = mean.unsqueeze(0) + std.unsqueeze(0) * eps          # (K, N, 15)
        samples_phys = (samples_std * train_ds.bin_stds.to(device)
                        + train_ds.bin_means.to(device))
        samples_NK15 = samples_phys.permute(1, 0, 2).cpu().numpy()        # (N, K, 15)

    keys_v, _ = _block_cond_concat(val_hcs, lit, cfg, train_ds)
    nlls, _ = _k_run_combined(val_hcs, keys_v, samples_NK15)
    return float(nlls.mean()), float(nlls.std()), val_gauss_nll


def score_checkpoint(name, cfg):
    lit, train_ds, val_ds, total_dim = _load_checkpoint(name, cfg)

    # Diffusion: K samples per val block, in physical residual units.
    keys_v, cond_v = _block_cond_concat(val_hcs, lit, cfg, train_ds)
    cond_K = repeat(cond_v, "n d -> (n k) d", k=K)
    rows_out = []

    def _diff_score_at_w(w):
        torch.manual_seed(0)
        samples = sample_conditional_extended(lit, cond_K, guidance_w=w, device=device).cpu().numpy()
        samples = samples.reshape(len(keys_v), K, 15)
        nlls, fl = _k_run_combined(val_hcs, keys_v, samples)
        return float(nlls.mean()), float(nlls.std()), float(fl.mean())

    if cfg.get("cond_dropout_p", 0.0) > 0.0:
        # CFG: sweep guidance values.
        for w in CFG_GUIDANCE_W:
            m, s, f = _diff_score_at_w(w)
            rows_out.append({"experiment": name, "guidance_w": w,
                             "nll_mean": m, "nll_std": s, "floor": f})
    else:
        m, s, f = _diff_score_at_w(0.0)
        rows_out.append({"experiment": name, "guidance_w": 0.0,
                         "nll_mean": m, "nll_std": s, "floor": f})

    # Oracle for this cond set.
    o_mean, o_std, gauss = _oracle_for(name, cfg, val_hcs, lit, train_ds, val_ds)
    for r in rows_out:
        r["oracle_nll_mean"] = o_mean
        r["oracle_nll_std"]  = o_std
        r["oracle_gauss"]    = gauss
        r["coverage"]        = float(det_cl_val["coverage"])
    return rows_out


all_rows = []
for name in _discovered:
    print(f"scoring {name} ...")
    all_rows.extend(score_checkpoint(name, EXPERIMENTS[name]))

scoreboard = pd.DataFrame(all_rows)
scoreboard["classical"] = nll_cl_val
print(scoreboard.to_string(index=False, float_format=lambda v: f"{v:.4f}"))


---
## Task 69 — Headline plot

Bar chart, val split: classical baseline plus every experiment's NLL,
with K-σ error bars. Oracle NLL per experiment overlaid as a
horizontal dashed marker to make the "what's achievable from this cond
set" boundary visible.

For E6 (CFG), the bar shown is the best-NLL guidance setting; a
secondary panel sweeps `w` so you can see the guidance vs NLL trade.


In [ ]:
# Task 69 — headline plot + CFG sweep (if E6 exists).

# Pick the best guidance for each experiment (lowest NLL).
best = (scoreboard.sort_values("nll_mean")
                  .groupby("experiment").head(1)
                  .sort_values("experiment").reset_index(drop=True))

fig, ax = plt.subplots(figsize=(max(8, 0.9 * len(best) + 4), 5))
x = np.arange(len(best) + 1)
labels = ["classical"] + list(best["experiment"])
heights = [nll_cl_val] + list(best["nll_mean"])
errors  = [0.0] + list(best["nll_std"])
colors  = ["C0"] + ["C2"] * len(best)

ax.bar(x, heights, yerr=errors, color=colors, edgecolor="black", linewidth=0.4)
for i, (xi, e) in enumerate(zip(x[1:], best["experiment"])):
    o = best.loc[i, "oracle_nll_mean"]
    ax.hlines(o, xi - 0.4, xi + 0.4, color="C3", linestyle="--", linewidth=1.5,
              label="oracle bound" if i == 0 else None)

ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20)
ax.set_ylabel("hard NLL (val, per-window)  —  lower is better")
ax.set_title("Diffusion experiments vs classical (with per-experiment oracle bound)")
ax.legend(fontsize=8); ax.axhline(nll_cl_val, color="C0", linestyle=":", linewidth=0.6)
plt.tight_layout(); plt.show()

# CFG sweep panel (only if E6 exists).
if "E6" in scoreboard["experiment"].values:
    sub = scoreboard[scoreboard["experiment"] == "E6"].sort_values("guidance_w")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(sub["guidance_w"], sub["nll_mean"], yerr=sub["nll_std"],
                marker="o", color="C2", label="E6 (CFG)")
    ax.axhline(nll_cl_val, color="C0", linestyle=":", linewidth=1.0, label="classical")
    if "E5" in scoreboard["experiment"].values:
        e5 = scoreboard[scoreboard["experiment"] == "E5"]["nll_mean"].iloc[0]
        ax.axhline(e5, color="C4", linestyle="-.", linewidth=1.0, label="E5 (no CFG)")
    ax.set_xlabel("guidance weight w"); ax.set_ylabel("hard NLL (val)")
    ax.set_title("Classifier-free guidance sweep")
    ax.legend(); plt.tight_layout(); plt.show()

# Per-hemicycle breakdown for the best variant — same axes as the
# Week-10 chart so improvement is visually unambiguous.
best_name = best.iloc[best["nll_mean"].idxmin()]["experiment"]
best_w    = float(best.iloc[best["nll_mean"].idxmin()]["guidance_w"])
print(f"per-hemicycle breakdown for best variant: {best_name} (guidance_w={best_w})")

lit, train_ds, val_ds, _ = _load_checkpoint(best_name, EXPERIMENTS[best_name])
keys_v, cond_v = _block_cond_concat(val_hcs, lit, EXPERIMENTS[best_name], train_ds)
cond_K = repeat(cond_v, "n d -> (n k) d", k=K)
torch.manual_seed(0)
samples = sample_conditional_extended(lit, cond_K, guidance_w=best_w, device=device).cpu().numpy()
samples = samples.reshape(len(keys_v), K, 15)

cycle_rows = []
for hc in val_hcs:
    nll_cl_hc, _ = hard_nll_classical(classical, [hc])
    nll_v_K, _   = _k_run_combined([hc], keys_v, samples)
    cycle_rows.append({"hc": f"{hc['cycle']:02d}{hc['hemisphere'][0]}",
                       "classical": nll_cl_hc,
                       f"{best_name}_mean": float(nll_v_K.mean()),
                       f"{best_name}_std":  float(nll_v_K.std())})
bdf = pd.DataFrame(cycle_rows)
fig, ax = plt.subplots(figsize=(min(15, 0.7 * len(bdf) + 4), 4.5))
x = np.arange(len(bdf)); w = 0.4
ax.bar(x - w/2, bdf["classical"],            width=w, color="C0", label="classical")
ax.bar(x + w/2, bdf[f"{best_name}_mean"], yerr=bdf[f"{best_name}_std"],
       width=w, color="C2", label=best_name)
ax.set_xticks(x); ax.set_xticklabels(bdf["hc"], rotation=30)
ax.set_ylabel("hard NLL (lower is better)")
ax.set_title(f"Per-hemicycle breakdown — per-window val ({best_name})")
ax.legend(); plt.tight_layout(); plt.show()


---
## Task 70 — Going further

If you've worked through E0–E7 and want to push further, the tiered
menu below ranks the next experiments by expected payoff per unit
effort. Discipline still applies: one knob at a time, log to wandb,
add to `EXPERIMENTS` and `_discovered`, then re-run this notebook.

**Level 1 — easy wins**
- **Wider/deeper MLP.** Bump `hidden_dim` from 128 to 256, `n_layers`
  from 3 to 5 in the winning experiment's config. If NLL drops, the
  network was capacity-bound — interesting on its own.
- **Longer K at evaluation.** Bump K from 100 to 500 for the winning
  variant — tightens the K-σ error bar and lets you trust smaller
  margins.
- **Sampler comparison.** Re-score the winner with DDPM (stochastic)
  sampling instead of the deterministic DDIM in
  `sample_conditional_extended`. Deterministic samplers can under-
  disperse, inflating NLL.

**Level 2 — extra cond information**
- **Larger trajectory K.** Bump `K_LAGS` from 4 to 8 in 10d. If E3's
  oracle improves but E3's diffusion doesn't, the architecture is
  underusing the longer history.
- **Lagged opposite-hemisphere.** Pair the trajectory cond with the
  opposite hemisphere — `opp_area_smoothed_lag1..lag4`.

**Level 3 — architectural changes**
- **Cross-attention conditioning.** Replace the FiLM mechanism with
  cross-attention over a small set of learned cond tokens — overkill
  for the cond dim here, but worth knowing if the FiLM gain saturates.
- **Per-bin-aware loss.** Weight the ε-prediction loss by the inverse
  per-bin std so well-resolved bins don't dominate gradients.

**The test set is the PI's.** Every iteration above is val-only. The
final test-set reveal happens once, after the program is closed.


---
## Handoff back to the PI

The headline numbers in `scoreboard` answer two questions:

1. **Does any variant beat the classical baseline on val?** If yes, the
   diffusion approach has earned its place in the final pipeline.
2. **Where is the bottleneck — information or architecture?** Compare
   each row's `nll_mean` to its `oracle_nll_mean`. A large gap means
   the cond set has more information than the diffusion is extracting
   (architecture-bound). A small gap with the oracle near classical
   means the cond set isn't carrying enough information — that line of
   experiments is exhausted; try a different cond group.

The PI will run the test set evaluation on whatever variant the val
results recommend.
